<a href="https://colab.research.google.com/github/raaghavkk/UG04-NLP-COMM061/blob/main/notebooks/akshyat_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Pre-trained Transformer Models
## COM3029 NLP Group Coursework


In [ ]:
# Install required packages
!pip install transformers datasets scikit-learn evaluate seqeval accelerate -q

In [ ]:

import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

# Two seed values used for two runs each
SEEDS = [42, 1234]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
# Load dataset and convert sentiment labels to integers
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")

train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

train_df["Sentiment"] = train_df["Sentiment"].astype(int)
val_df["Sentiment"] = val_df["Sentiment"].astype(int)
test_df["Sentiment"] = test_df["Sentiment"].astype(int)

print(f"Data loaded. Training rows: {len(train_df)}")

In [ ]:
# Format dataset and tokenise text
from datasets import Dataset

def tokenise_data(df, tokenizer):
    df_subset = df[["text", "Sentiment"]]
    df_subset = df_subset.rename(columns={"Sentiment": "labels"})

    ds = Dataset.from_pandas(df_subset)

    def tokenise(batch):
        return tokenizer(batch["text"], truncation=True, max_length=128)

    ds = ds.map(tokenise, batched=True)
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")

    return ds

print("Tokenisation function ready")

In [ ]:
# Set up the F1 score calculation for the training loop
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

print("F1 metric ready")


## Cell 6 Training Helper
Single function that trains a model, evaluates on the test set, and returns results.
Called for each run and each model.

In [ ]:
# Set up the training loop to fine-tune the model and test it
def train_and_evaluate(model_name, train_df, val_df, test_df, seed=42):
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Convert our data using the function we made earlier
    train_dataset = tokenise_data(train_df, tokenizer)
    val_dataset = tokenise_data(val_df, tokenizer)
    test_dataset = tokenise_data(test_df, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(DEVICE)

    run_dir = f"./results_{seed}"

    training_args = TrainingArguments(
        output_dir=run_dir,
        num_train_epochs=3, # keeping this at 3 so Colab doesn't crash
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=torch.cuda.is_available(),
        seed=seed,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Test the model and get the results
    raw_preds = trainer.predict(test_dataset)
    preds = np.argmax(raw_preds.predictions, axis=-1)
    labels = raw_preds.label_ids

    report = classification_report(
        labels, preds,
        target_names=["Negative", "Positive"],
        output_dict=True
    )

    print(classification_report(labels, preds, target_names=["Negative", "Positive"]))
    print(f"Macro-F1: {report['macro avg']['f1-score']:.4f}")

    return report, preds, labels, model, tokenizer

print("Training function ready")


# Section 2.1 Baseline vs Fine-tuned Transformer (Sentiment Task)

**requirement**: Compare classical TF-IDF + LR against RoBERTa-base fine-tuned on Sentiment.



In [ ]:
# Train our first RoBERTa model
roberta_report_1, roberta_preds_1, roberta_labels_1, roberta_model_1, _ = train_and_evaluate(
    "roberta-base", train_df, val_df, test_df, seed=42
)

In [ ]:
# Run 2: RoBERTa-base (seed 1234)
roberta_report_2, roberta_preds_2, roberta_labels_2, roberta_model_2, roberta_tokenizer = train_and_evaluate(
    "roberta-base", train_df, val_df, test_df, seed=1234
)

In [ ]:
# Section 2.1 Summary Table
# TF-IDF + LR results (seed 42 and 1234 both runs identical, confirming stability)
# LR Sentiment Macro-F1: 0.84 | Precision: 0.84 | Recall: 0.84

r1 = roberta_report_1["macro avg"]
r2 = roberta_report_2["macro avg"]

avg_f1 = (r1["f1-score"] + r2["f1-score"]) / 2

summary_df = pd.DataFrame({
    "Model": ["LR (Baseline)", "LR (Baseline)", "RoBERTa", "RoBERTa"],
    "Run": [1, 2, 1, 2],
    "Seed": [42, 1234, 42, 1234],
    "Macro-F1": [0.84, 0.84, round(r1['f1-score'], 4), round(r2['f1-score'], 4)],
    "Precision": [0.84, 0.84, round(r1['precision'], 4), round(r2['precision'], 4)],
    "Recall": [0.84, 0.84, round(r1['recall'], 4), round(r2['recall'], 4)]
})

print("Results for report:")
print(summary_df)
print(f"\nRoBERTa Avg F1: {round(avg_f1, 4)}")
print(f"Improvement: +{round(avg_f1 - 0.84, 4)}")

In [ ]:
# Confusion Matrix for best RoBERTa run
# Use run 2 (last run) swap to run 1 if it scores higher
cm = confusion_matrix(roberta_labels_2, roberta_preds_2)
ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(cmap="Blues")

plt.title("RoBERTa Sentiment")
plt.savefig("cm_roberta_sentiment.png")
plt.show()


# Section 2.3  Monolingual vs Multilingual

 Compare RoBERTa-base (English only) against XLM-RoBERTa-base




In [ ]:
# Run 1: XLM-RoBERTa-base, Seed=42
xlm_report_1, xlm_preds_1, xlm_labels_1, xlm_model_1, _ = train_and_evaluate(
    "xlm-roberta-base", train_df, val_df, test_df, seed=42
)

In [ ]:
# Run 2: XLM-RoBERTa-base, Seed=1234
xlm_report_2, xlm_preds_2, xlm_labels_2, xlm_model_2, xlm_tokenizer = train_and_evaluate(
    "xlm-roberta-base", train_df, val_df, test_df, seed=1234
)

In [ ]:
# Check Macro-F1 for each variety to see if XLM helps with Indian English
test_df['roberta_pred'] = roberta_preds_2
test_df['xlm_pred'] = xlm_preds_2

print("Variety | RoBERTa F1 | XLM F1")
print("-" * 30)

for v in ["en-AU", "en-UK", "en-IN"]:
    subset = test_df[test_df["variety"] == v]

    y_true = subset["Sentiment"].values
    y_rob = subset["roberta_pred"].values
    y_xlm = subset["xlm_pred"].values

    f1_rob = f1_score(y_true, y_rob, average="macro")
    f1_xlm = f1_score(y_true, y_xlm, average="macro")

    print(f"{v} | {f1_rob:.4f} | {f1_xlm:.4f}")

In [ ]:
# Section 2.3 Summary Table
r1_xlm = xlm_report_1["macro avg"]
r2_xlm = xlm_report_2["macro avg"]
r1_rob = roberta_report_1["macro avg"]
r2_rob = roberta_report_2["macro avg"]

avg_xlm = (r1_xlm["f1-score"] + r2_xlm["f1-score"]) / 2
avg_rob = (r1_rob["f1-score"] + r2_rob["f1-score"]) / 2

summary_23 = pd.DataFrame({
    "Model": ["RoBERTa", "RoBERTa", "XLM-R", "XLM-R"],
    "Run": [1, 2, 1, 2],
    "Seed": [42, 1234, 42, 1234],
    "Macro-F1": [round(r1_rob['f1-score'], 4), round(r2_rob['f1-score'], 4),
                 round(r1_xlm['f1-score'], 4), round(r2_xlm['f1-score'], 4)],
    "Precision": [round(r1_rob['precision'], 4), round(r2_rob['precision'], 4),
                  round(r1_xlm['precision'], 4), round(r2_xlm['precision'], 4)],
    "Recall": [round(r1_rob['recall'], 4), round(r2_rob['recall'], 4),
               round(r1_xlm['recall'], 4), round(r2_xlm['recall'], 4)]
})

print("Section 2.3 Results:")
print(summary_23)
print(f"\nRoBERTa Avg F1: {round(avg_rob, 4)}")
print(f"XLM-RoBERTa Avg F1: {round(avg_xlm, 4)}")

In [ ]:
# Compare confusion matrices for Mono vs Multilingual models
plt.figure(figsize=(10, 4))

# RoBERTa Plot
ax1 = plt.subplot(1, 2, 1)
cm_rob = confusion_matrix(roberta_labels_2, roberta_preds_2)
ConfusionMatrixDisplay(cm_rob, display_labels=["Neg", "Pos"]).plot(cmap="Blues", ax=ax1)
plt.title("RoBERTa (Mono)")

# XLM-R Plot
ax2 = plt.subplot(1, 2, 2)
cm_xlm = confusion_matrix(xlm_labels_2, xlm_preds_2)
ConfusionMatrixDisplay(cm_xlm, display_labels=["Neg", "Pos"]).plot(cmap="Greens", ax=ax2)
plt.title("XLM-R (Multi)")

plt.tight_layout()
plt.savefig("cm_comparison_23.png")
plt.show()


## Visual Comparison All Experiments
Bar chart showing Macro-F1 across all models.

In [ ]:
# Compare F1 scores for all models
models = ['LR R1', 'LR R2', 'RoBERTa R1', 'RoBERTa R2', 'XLM-R R1', 'XLM-R R2']

# Pulling the F1 scores from our previous reports
f1_vals = [
    0.84, 0.84,
    roberta_report_1["macro avg"]["f1-score"],
    roberta_report_2["macro avg"]["f1-score"],
    xlm_report_1["macro avg"]["f1-score"],
    xlm_report_2["macro avg"]["f1-score"]
]

plt.figure(figsize=(10, 5))
bars = plt.bar(models, f1_vals, color=['lightblue', 'lightblue', 'royalblue', 'navy', 'orange', 'darkorange'])

# Add the actual numbers on top of the bars
plt.bar_label(bars, fmt='%.3f', padding=3)

plt.ylim(0, 1.1)
plt.ylabel("Macro-F1")
plt.title("Sentiment Classification Performance")

# Show random chance line
plt.axhline(0.5, color="red", linestyle="--", label="Random (0.5)")
plt.legend()

plt.savefig("f1_comparison.png")
plt.show()


## Save Best Model
Saving the best RoBERTa model



In [ ]:
# Save the best RoBERTa model for the report and deployment
if r1["f1-score"] >= r2["f1-score"]:
    best_model = roberta_model_1
    best_name = "run1"
else:
    best_model = roberta_model_2
    best_name = "run2"

save_path = f"./best_roberta_{best_name}"
best_model.save_pretrained(save_path)
roberta_tokenizer.save_pretrained(save_path)

print(f"Model saved to {save_path}")
# @Team: use this for the LIME/SHAP analysis and the web app.
# REMINDER: do not include this folder in the final zip upload!